In [69]:
import pandas as pd
from datasets import load_dataset

In [70]:
df1= pd.read_csv('Google_mistral_meta.csv') #Casimiro

df2 = pd.read_csv('article_level_data.csv') #Beatriz

df3= pd.read_csv('train.csv') #Beatriz

In [71]:
print(df1['model'].value_counts())

model
Google     4143
Mistral    4143
Meta       4143
Name: count, dtype: int64


In [72]:
print(df2['class'].value_counts())

class
0    509
1    509
Name: count, dtype: int64


In [73]:
df2_renamed = df2[['article', 'class']].rename(columns={'article': 'content'})
df2_renamed['model'] = df2_renamed['class'].map({0: 'Human', 1: 'OpenAI'})
df2_renamed = df2_renamed.drop(columns=['class'])

df_final = pd.concat([df1, df2_renamed], ignore_index=True)


In [74]:
print(df_final['model'].value_counts())

model
Google     4143
Mistral    4143
Meta       4143
Human       509
OpenAI      509
Name: count, dtype: int64


In [75]:
print(df3.columns)

Index(['source', 'id', 'text'], dtype='str')


In [76]:
print(df3['source'].value_counts())

source
human    198932
Name: count, dtype: int64


In [77]:
df3_renamed = df3[['text']].rename(columns={'text': 'content'})
df3_renamed['model'] = 'Human'
df3_renamed = df3_renamed.sample(n=20000, random_state=42)

df_final = pd.concat([df_final, df3_renamed], ignore_index=True)
print(df_final['model'].value_counts())


model
Human      20509
Google      4143
Mistral     4143
Meta        4143
OpenAI       509
Name: count, dtype: int64


In [78]:
print(len(df_final))

33447


In [79]:
df4=pd.read_json('train.json')

In [80]:
df4.columns

Index(['url', 'content', 'model'], dtype='str')

In [81]:
df4['model'].value_counts()

model
meta-llama/Llama-3.1-8B-Instruct      33140
Qwen/Qwen2.5-7B-Instruct              33140
mistralai/Mistral-7B-Instruct-v0.3    33140
google/gemma-2-9b-it                  33140
Intel/neural-chat-7b-v3-3             33140
microsoft/Phi-3.5-mini-instruct       33140
upstage/SOLAR-10.7B-Instruct-v1.0     33140
Name: count, dtype: int64

In [82]:
model_map = {
    'meta-llama/Llama-3.1-8B-Instruct': 'Meta',
    'google/gemma-2-9b-it': 'Google',
}

df4_renamed = df4[df4['model'].isin(model_map)].copy()
df4_renamed['model'] = df4_renamed['model'].map(model_map)

print(df4_renamed['model'].value_counts())


model
Meta      33140
Google    33140
Name: count, dtype: int64


In [83]:
# Remove linhas Mistral existentes e adiciona Meta/Google do df4
df_final = df_final[~df_final['model'].astype(str).str.contains('mistral', case=False, na=False)].copy()
df_final = pd.concat([df_final, df4_renamed[['content', 'model']]], ignore_index=True)

# Carregar dataset Hugging Face e extrair 10k linhas Claude/Anthropic
dataset = load_dataset('ahmadreza13/human-vs-Ai-generated-dataset', split='train')
hf_df = dataset.to_pandas()

def extract_content_from_data(value):
    # O campo 'data' pode vir como texto, dict, lista ou JSON serializado.
    if pd.isna(value):
        return None

    if isinstance(value, str):
        s = value.strip()
        if not s:
            return None
        # Se for JSON serializado, tenta extrair texto interno.
        if (s.startswith('{') and s.endswith('}')) or (s.startswith('[') and s.endswith(']')):
            import json
            try:
                parsed = json.loads(s)
                return extract_content_from_data(parsed)
            except Exception:
                return s
        return s

    if isinstance(value, dict):
        for key in ['text', 'content', 'prompt', 'response', 'output']:
            if key in value and isinstance(value[key], str) and value[key].strip():
                return value[key].strip()
        for v in value.values():
            extracted = extract_content_from_data(v)
            if extracted:
                return extracted
        return None

    if isinstance(value, list):
        parts = []
        for item in value:
            extracted = extract_content_from_data(item)
            if extracted:
                parts.append(extracted)
        return ' '.join(parts).strip() if parts else None

    return str(value).strip() if str(value).strip() else None

# Filtro Claude/Anthropic usando coluna 'model' (schema reportado: data/generated/model).
claude_mask = hf_df['model'].astype(str).str.contains(r'claude|anthropic', case=False, na=False)

if not claude_mask.any():
    raise ValueError("Nao encontrei entradas Claude/Anthropic na coluna 'model'.")

if 'data' in hf_df.columns:
    content_series = hf_df['data'].apply(extract_content_from_data)
elif 'text' in hf_df.columns:
    content_series = hf_df['text'].astype(str)
elif 'content' in hf_df.columns:
    content_series = hf_df['content'].astype(str)
else:
    raise ValueError(f"Nao encontrei coluna de texto no dataset. Colunas disponiveis: {hf_df.columns.tolist()}")

df_claude = pd.DataFrame({'content': content_series, 'model': 'Anthropic'})
df_claude = df_claude[claude_mask].dropna(subset=['content']).copy()
df_claude['content'] = df_claude['content'].astype(str).str.strip()
df_claude = df_claude[df_claude['content'] != '']

if len(df_claude) < 10000:
    raise ValueError(f"So existem {len(df_claude)} linhas Claude/Anthropic apos filtro; nao da para amostrar 10000 sem reposicao.")

df_claude_10000 = df_claude.sample(n=10000, random_state=42).reset_index(drop=True)

df_final = pd.concat([df_final, df_claude_10000[['content', 'model']]], ignore_index=True)
print(df_final['model'].value_counts())
print(len(df_final))

model
Google       37283
Meta         37283
Human        20509
Anthropic    10000
OpenAI         509
Name: count, dtype: int64
105584


In [84]:
df_final['model'].value_counts()

model
Google       37283
Meta         37283
Human        20509
Anthropic    10000
OpenAI         509
Name: count, dtype: int64

In [85]:
len(df_final)

105584

In [86]:
df5 = pd.read_json('all.jsonl', lines=True)

In [87]:
df5

,question,human_answers,chatgpt_answers,index,source
0,"Why is every book I hear about a "" NY Times # ...","[Basically there are many categories of "" Best...",[There are many different best seller lists th...,NaN,reddit_eli5
1,"If salt is so bad for cars , why do we use it ...",[salt is good for not dying in car crashes and...,[Salt is used on roads to help melt ice and sn...,NaN,reddit_eli5
2,Why do we still have SD TV channels when HD lo...,[The way it works is that old TV stations got ...,[There are a few reasons why we still have SD ...,NaN,reddit_eli5
3,Why has nobody assassinated Kim Jong - un He i...,[You ca n't just go around assassinating the l...,[It is generally not acceptable or ethical to ...,NaN,reddit_eli5
4,How was airplane technology able to advance so...,[Wanting to kill the shit out of Germans drive...,[After the Wright Brothers made the first powe...,NaN,reddit_eli5
...,...,...,...,...,...
24317,Is rise in pressure from 116/66 to 140/80 norm...,[Hello!Welcome and thank you for asking on HCM...,[It's not uncommon for blood pressure to fluct...,NaN,medicine
24318,What could cause a painless lump in the right ...,"[Hi, * As per my surgical experience, the issu...",[There are several possible causes of a painle...,NaN,medicine
24319,Can Acutret be given to a child for treatment ...,[Although it is difficult to comment whether A...,[It is not appropriate for me to recommend a s...,NaN,medicine
24320,Are BP of 119/65 and pulse of 35 causes for co...,[Welcome and thank you for asking on HCM! I ha...,[It is not uncommon for people with rheumatoid...,NaN,medicine


In [88]:
df5_openai = df5[['chatgpt_answers']].copy()
df5_openai['model'] = 'OpenAI'

df5_openai.head()

,chatgpt_answers,model
0,[There are many different best seller lists th...,OpenAI
1,[Salt is used on roads to help melt ice and sn...,OpenAI
2,[There are a few reasons why we still have SD ...,OpenAI
3,[It is generally not acceptable or ethical to ...,OpenAI
4,[After the Wright Brothers made the first powe...,OpenAI


In [89]:
df_final = pd.concat([df_final, df5_openai], ignore_index=True)
print(df_final['model'].value_counts())
print(len(df_final))

model
Google       37283
Meta         37283
OpenAI       24831
Human        20509
Anthropic    10000
Name: count, dtype: int64
129906


In [90]:
df_human_20000 = (
    df_final[df_final['model'] == 'Human']
    .sample(n=20000, random_state=42)
    .reset_index(drop=True)
)

df_non_human_10000 = (
    df_final[df_final['model'] != 'Human']
    .groupby('model', group_keys=False)
    .sample(n=10000, random_state=42)
    .reset_index(drop=True)
)

df_final_filtered = pd.concat([df_human_20000, df_non_human_10000], ignore_index=True)

print(df_final_filtered['model'].value_counts())
print(len(df_final_filtered))

model
Human        20000
Anthropic    10000
Google       10000
Meta         10000
OpenAI       10000
Name: count, dtype: int64
60000


In [91]:
df_final_filtered.to_csv('df_final.csv', index=False)

In [ ]:
import os
import re
import unicodedata

MAX_WORDS = 120
MIN_CHARS = 100

# Função de limpeza leve: normaliza texto, remove caracteres invisíveis, normaliza espaços e quebras de linha.
def clean_text_light(text):
    text = str(text)
    text = unicodedata.normalize('NFKC', text)
    
    # Normaliza aspas e travessoes comuns para ASCII simples
    replacements = {
        '\u2018': "'",
        '\u2019': "'",
        '\u201c': '"',
        '\u201d': '"',
        '\u2013': '-',
        '\u2014': '-',
        '\u00a0': ' ',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    # Remove caracteres de controlo invisiveis, preservando \n e \t
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)

    # Normaliza espacos e quebras de linha em excesso
    text = re.sub(r'\r\n?', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

#função que remove aspras e parenteses no inicio e no fim pois são vestigios de json ou formatações antigas, e podem atrapalhar o modelo.
def strip_json_edge_artifacts(text):
    text = str(text).strip()

    # Remove repetidamente vestigios no inicio/fim: aspas, parenteses e colchetes.
    edge_chars = '"\'()[]{}'
    while text and text[0] in edge_chars:
        text = text[1:].lstrip()
    while text and text[-1] in edge_chars:
        text = text[:-1].rstrip()

    return text

# Função para truncar texto respeitando limites de palavras e tentando manter integridade de sentenças.
def truncate_sentence_boundary_by_words(text, max_words=120):
    text = " ".join(str(text).split())
    words = text.split()
    if len(words) <= max_words:
        return text

    cutoff = " ".join(words[:max_words])
    # Procura o ultimo fim de frase dentro da janela.
    boundary = max(cutoff.rfind('.'), cutoff.rfind('!'), cutoff.rfind('?'))
    if boundary >= int(len(cutoff) * 0.6):
        return cutoff[:boundary + 1].strip()

    # Fallback: corta em max_words para evitar partir texto no meio.
    return cutoff.strip()

df_ready = df_final_filtered.copy()
if 'chatgpt_answers' in df_ready.columns:
    df_ready['content'] = df_ready['content'].fillna(df_ready['chatgpt_answers'])

df_ready = df_ready[['content', 'model']].dropna(subset=['content']).copy()
df_ready['content'] = df_ready['content'].apply(clean_text_light)
df_ready['content'] = df_ready['content'].apply(strip_json_edge_artifacts)
df_ready['content'] = df_ready['content'].apply(lambda x: truncate_sentence_boundary_by_words(x, MAX_WORDS))
df_ready['n_chars'] = df_ready['content'].str.len()

# Mantem textos >= 100 chars; os >120 palavras ja foram truncados.
df_len_filtered = df_ready[df_ready['n_chars'] >= MIN_CHARS].copy()

# Split 70/15/15 estratificado por classe.
parts = []
for model_name, group in df_len_filtered.groupby('model'):
    g = group.sample(frac=1, random_state=42).reset_index(drop=True)
    n = len(g)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    n_test = n - n_train - n_val

    train_part = g.iloc[:n_train]
    val_part = g.iloc[n_train:n_train + n_val]
    test_part = g.iloc[n_train + n_val:]
    parts.append((train_part, val_part, test_part))

train_df = pd.concat([p[0] for p in parts], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
val_df = pd.concat([p[1] for p in parts], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
test_df = pd.concat([p[2] for p in parts], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

train_df = train_df[['content', 'model']]
val_df = val_df[['content', 'model']]
test_df = test_df[['content', 'model']]

os.makedirs('splits', exist_ok=True)
train_df.to_csv('splits/train.csv', index=False)
val_df.to_csv('splits/val.csv', index=False)
test_df.to_csv('splits/test.csv', index=False)

print('Train:', len(train_df))
print('Val:', len(val_df))
print('Test:', len(test_df))
print('Total apos truncar+filtrar:', len(df_len_filtered))
print('\nDistribuicao por model:')
print(df_len_filtered['model'].value_counts())
print('\nResumo de tamanhos (chars):')
print(df_len_filtered['n_chars'].describe().round(2))

Train: 41809
Val: 8958
Test: 8962
Total apos truncar+filtrar: 59729

Distribuicao por model:
model
Human        20000
Google       10000
Meta          9999
Anthropic     9936
OpenAI        9794
Name: count, dtype: int64

Resumo de tamanhos (chars):
count    59729.0
mean       667.3
std         94.8
min        102.0
25%        620.0
50%        676.0
75%        727.0
max       2043.0
Name: n_chars, dtype: float64


In [93]:
# Diagnostico: distribuicao de contagem de palavras apos truncagem
n_words = df_len_filtered['content'].str.split().str.len()
print('Resumo de tamanhos (palavras):')
print(n_words.describe().round(2))
print(f'\nTextos com <= {MAX_WORDS} palavras: {(n_words <= MAX_WORDS).sum()} / {len(n_words)} ({100*(n_words <= MAX_WORDS).mean():.1f}%)')
print(f'Textos com == {MAX_WORDS} palavras (truncados exactamente): {(n_words == MAX_WORDS).sum()}')
print('\nDistribuicao por modelo (media de palavras):')
print(df_len_filtered.copy().assign(n_words=n_words).groupby('model')['n_words'].mean().round(1))

Resumo de tamanhos (palavras):
count    59729.00
mean       107.26
std         12.23
min          1.00
25%        103.00
50%        110.00
75%        115.00
max        120.00
Name: content, dtype: float64

Textos com <= 120 palavras: 59729 / 59729 (100.0%)
Textos com == 120 palavras (truncados exactamente): 3409

Distribuicao por modelo (media de palavras):
model
Anthropic    107.8
Google       110.0
Human        105.7
Meta         108.1
OpenAI       106.2
Name: n_words, dtype: float64


In [94]:
train_df.head()

,content,model
0,Key Attestation gives you more confidence that...,Human
1,"Join us for an evening of sharing poetry, stor...",Human
2,Skulls Found In Mexico May Have Belonged To Hu...,Google
3,Submarines are made of very strong materials l...,OpenAI
4,9 Ways Dating And Working Out Are Similar Febr...,Meta
